# Paper demo: sparse reconstruction

The paper's Figure 7 measures how well the largest ICA coordinates reconstruct held-out residual-stream activations. This lightweight demo applies the same top-k cosine calculation to the exact financial-context sentence used in Figure 24. It demonstrates the metric; it is not a replacement for the paper's corpus-level benchmark.

In [ ]:
%pip install -q icalens==0.4.0 matplotlib

In [ ]:
from icalens import ICALens

lens = ICALens.from_pretrained(
    "sida/icalens-gpt2-small-pile10k",
)
text = (
    "Maya stopped at the bank before the trip, waiting in line to deposit "
    "a check and withdraw enough cash for the weekend."
)
result = lens.analyze(text, layer=6)
result

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

ks = [1, 3, 10, 32, 100, 300, result.scores.shape[-1]]
component_count = result.scores.shape[-1]
zero_scores = result.scores.new_zeros((1, component_count))
center = lens.inverse_transform(zero_scores, layer=6)[0]
basis_scores = result.scores.new_zeros((component_count, component_count))
basis_scores.fill_diagonal_(1)
writing_directions = lens.inverse_transform(basis_scores, layer=6) - center
direction_norms = writing_directions.norm(dim=-1)
mean_cosines = []
for k in ks:
    contribution = result.scores.abs() * direction_norms
    indices = contribution.topk(k, dim=-1).indices
    top_scores = result.scores.new_zeros(result.scores.shape)
    top_scores.scatter_(1, indices, result.scores.gather(1, indices))
    reconstruction = lens.inverse_transform(top_scores, layer=6)
    similarity = F.cosine_similarity(result.activations, reconstruction, dim=-1)
    mean_cosines.append(similarity.mean().item())

plt.figure(figsize=(7, 4))
plt.plot(ks, mean_cosines, marker="o", color="#136f63")
plt.xscale("log")
plt.ylim(0, 1.02)
plt.xlabel("Retained ICA coordinates per token (k)")
plt.ylabel("Mean reconstruction cosine")
plt.title("GPT-2 Small, layer 6: paper example sentence")
plt.grid(alpha=0.25)
plt.show()

The component set is selected independently at each token by `abs(score) * writing-direction norm`, matching the accepted paper experiment in `experiments/reconstruction/official`. The budgets are the paper values 1, 3, 10, 32, 100, and 300, plus the complete 768-coordinate endpoint. The full benchmark uses six held-out domains across all model layers and compares ICA with matched SAE, PCA, and random baselines.